# MetalVision AI — Treinar no Google Colab
Este notebook instala o projeto e tenta baixar a base NEU-CLS do Kaggle. O resultado é um modelo experimental: **não valida inspeções de fotografias de celular**.

Se o download automático falhar, faça upload do ZIP da base NEU e descompacte no Colab.


In [ ]:
!git clone https://github.com/Vinicius-Santoss-silva/metalvision-ai.git
%cd metalvision-ai
!pip -q install -r requirements.txt kagglehub


In [ ]:
# Tenta baixar a base pública. Caso o Kaggle solicite credenciais, use a célula alternativa.
from pathlib import Path
import kagglehub
try:
    dataset_dir = Path(kagglehub.dataset_download('kaustubhdikshit/neu-surface-defect-database'))
    print('Dados encontrados em:', dataset_dir)
except Exception as err:
    dataset_dir = None
    print('Download automático indisponível:', err)
    print('Siga a célula de upload manual abaixo.')


In [ ]:
# ALTERNATIVA: execute somente se dataset_dir for None ou o treino não identificar as classes.
# Baixe manualmente o ZIP da base NEU-CLS e faça upload aqui.
if dataset_dir is None:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    archive = next(iter(uploaded))
    dataset_dir = Path('/content/neu_dataset')
    dataset_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        # Os arquivos são fornecidos pelo usuário; não extraia ZIPs de origem desconhecida.
        for member in z.infolist():
            destination = (dataset_dir / member.filename).resolve()
            if not destination.is_relative_to(dataset_dir.resolve()):
                raise ValueError('ZIP contém caminho inválido.')
        z.extractall(dataset_dir)
print('Pasta do dataset:',dataset_dir)


In [ ]:
# Treinamento e avaliação: separação estratificada 80/20 em imagens NEU.
from training.train import train
metrics = train(dataset_dir, 'models')
print('Acurácia no teste NEU:', round(metrics['accuracy']*100,2),'%')


In [ ]:
# Baixe os dois resultados e envie para /models no GitHub.
from google.colab import files
files.download('models/metalvision_svm.joblib')
files.download('models/metricas.json')


## Publicação
No repositório GitHub, abra a pasta **models/**, escolha **Add file → Upload files** e envie os dois arquivos baixados. Após o Streamlit atualizar, a aba de análise passa a mostrar classificações.

**Cuidados:** o modelo conhece apenas seis classes com defeitos, não uma classe sem defeitos. Confiança do SVM não garante acerto nem generalização para imagens fora do conjunto NEU.
